# 01-EDA：客户流失数据探索

本笔记本对电信客户流失数据集进行探索性数据分析。

In [ ]:
import sys
from pathlib import Path

# 设置路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_utils import load_config
from src.viz_utils import plot_corr_heatmap, plot_distribution

# 加载配置
config = load_config(project_root / "config.yaml")
config

In [ ]:
# 加载原始数据
from src.data_prep import load_data_from_hf

df_raw = load_data_from_hf(config["data"]["dataset-name"])
df_raw.head()

In [ ]:
# 基础统计信息
df_raw.info()
df_raw.describe(include="all")

In [ ]:
# 目标变量分布
df_raw["Churn"].value_counts(normalize=True)

In [ ]:
# 类别特征与流失的关联
categorical_cols = df_raw.select_dtypes(include=["object"]).columns.drop(["customerID"], errors="ignore")

fig, axes = plt.subplots(4, 4, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(categorical_cols[:16]):
    pd.crosstab(df_raw[col], df_raw["Churn"], normalize="index").plot(
        kind="bar", ax=axes[i], title=col, rot=45
    )

plt.tight_layout()
plt.show()

In [ ]:
# 数值特征分布
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(numeric_cols):
    for churn, color in zip(["No", "Yes"], ["steelblue", "coral"]):
        subset = df_raw[df_raw["Churn"] == col]
        axes[i].hist(subset[col].dropna(), bins=30, alpha=0.6, label=churn, color=color)
    axes[i].set_title(f"{col} 分布")
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 数值特征相关性
plot_corr_heatmap(df_raw.select_dtypes(include=["number"]))